# Sports Highlight Generator — AMD ROCm Notebook (Fresh Build)
### TCS x AMD Hackathon · Track 2 (Multimodal) · MULTIMODAL_003

**What this notebook produces:** a single `highlight_reel.mp4` where each detected
highlight clip has (a) a virtual advertisement composited onto a playing surface and
(b) an AI-written social caption burned in.

**Pipeline (unchanged architecture):**
3 parallel excitement signals → weighted fusion → peak detection → clip extraction →
virtual ad insertion → LLM caption → stitch.

---
#### What changed vs. the previous notebook (and why it now runs on AMD)

| Previous (fragile) | This build (robust) | Reason |
|---|---|---|
| vLLM background server + **gated Llama 3.1 8B** | Caption LLM loaded **in-process via `transformers`** using **open** `Qwen2.5-3B-Instruct` (auto-fallback to 1.5B → template) | No background subprocess, no gated download, no health-poll timeouts |
| `archive.org` placeholder URL / SoccerNet download | Use your own file **or** auto-generate a synthetic test clip with planted "roar" moments | Notebook always runs end-to-end, even offline |
| EfficientNet faking `SIX_HIT` labels | Stream C = **motion energy + scene-cut** detection (real signal, no fake label map) | Genuine, defensible visual signal; no model download needed |
| Hard crashes on missing pieces | Every stage has graceful fallbacks | A demo should degrade, not die |

Everything still runs on AMD because PyTorch maps **ROCm → the CUDA API**, so
`torch.cuda.is_available()` returns `True` on Instinct GPUs and no AI code changes.


## 1 · Environment check (GPU / ROCm + ffmpeg)

In [ ]:
import shutil, subprocess, sys, os

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# ---- Ensure ffmpeg + ffprobe are available (the whole pipeline needs them) ----
def ensure_ffmpeg():
    if shutil.which("ffmpeg") and shutil.which("ffprobe"):
        return True
    print("ffmpeg/ffprobe not found -> installing...")

    # Method 1: apt (works on AMD Notebooks, which run as root)
    sh("apt-get update -y")
    sh("apt-get install -y ffmpeg")
    if shutil.which("ffmpeg") and shutil.which("ffprobe"):
        print("Installed ffmpeg via apt."); return True

    # Method 2: no-sudo fallback -- static binaries via pip, added to PATH
    print("apt unavailable -> trying static-ffmpeg (pip)...")
    sh(f"{sys.executable} -m pip install -q static-ffmpeg")
    try:
        import static_ffmpeg
        static_ffmpeg.add_paths()        # puts ffmpeg AND ffprobe on PATH for this kernel
    except Exception as e:
        print("static-ffmpeg failed:", str(e)[:120])

    # Method 3: conda, if present
    if not (shutil.which("ffmpeg") and shutil.which("ffprobe")) and shutil.which("conda"):
        print("trying conda...")
        sh("conda install -y -c conda-forge ffmpeg")

    return bool(shutil.which("ffmpeg") and shutil.which("ffprobe"))

ff_ok = ensure_ffmpeg()

print("Python :", sys.version.split()[0])

try:
    import torch
    print("torch  :", torch.__version__)
    # On AMD, torch.version.hip is set; torch.version.cuda is usually None.
    print("ROCm/HIP build :", getattr(torch.version, "hip", None))
    avail = torch.cuda.is_available()
    print("GPU available  :", avail)
    if avail:
        print("GPU name       :", torch.cuda.get_device_name(0))
        free, total = torch.cuda.mem_get_info()
        print(f"GPU VRAM       : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
    DEVICE = "cuda" if avail else "cpu"
except Exception as e:
    print("torch not importable yet:", e)
    DEVICE = "cpu"

if ff_ok:
    print("ffmpeg :", shutil.which("ffmpeg"))
    print("ffprobe:", shutil.which("ffprobe"))
else:
    print("ffmpeg : STILL MISSING -- if your environment is offline, run "
          "'apt-get install -y ffmpeg' manually or load an ffmpeg module.")

print("\nDEVICE selected ->", DEVICE)
print("\nIf GPU available is False on AMD Notebooks, switch the runtime to "
      "'Default (ROCm + vLLM)' from the launcher and re-run this cell.")

# ---- AMD/ROCm fix: torchvision's compiled NMS op has no GPU(HIP) kernel in
# this environment (only CPU is registered) -- see the NotImplementedError
# 'Could not run torchvision::nms ... CUDA backend'. Reroute just the NMS
# step through CPU (it only processes a small candidate-box list per image,
# so the cost is negligible) instead of hunting for a ROCm-matched wheel.
def _patch_nms_for_rocm():
    try:
        import torchvision
        if getattr(torchvision.ops.nms, "_rocm_cpu_patched", False):
            return
        _orig_nms = torchvision.ops.nms
        def _cpu_safe_nms(boxes, scores, iou_threshold):
            if boxes.is_cuda:
                keep = _orig_nms(boxes.cpu(), scores.cpu(), iou_threshold)
                return keep.to(boxes.device)
            return _orig_nms(boxes, scores, iou_threshold)
        _cpu_safe_nms._rocm_cpu_patched = True
        torchvision.ops.nms = _cpu_safe_nms
        print("Patched torchvision.ops.nms -> CPU fallback (ROCm NMS kernel workaround)")
    except Exception as e:
        print("nms patch skipped:", str(e)[:100])

_patch_nms_for_rocm()


## 2 · Install dependencies
AMD Notebooks already ships PyTorch built against ROCm — we do **not** reinstall it.
We only add the audio / vision / model libraries on top. `ultralytics` (player
occlusion) and `pytesseract` (scoreboard OCR) are optional and safe to skip.

In [ ]:
# Core libs needed by the pipeline. PyTorch is intentionally NOT reinstalled.
%pip install -q librosa soundfile transformers accelerate timm \
    opencv-python-headless scipy matplotlib Pillow

# ultralytics powers BOTH the trained hoarding detector and player/ball occlusion:
%pip install -q ultralytics
# OPTIONAL (cricket scoreboard OCR): %pip install -q pytesseract  (+ apt-get install tesseract-ocr)
# (pytesseract also needs the system binary: apt-get install -y tesseract-ocr)

print("Dependencies installed.")


## 3 · Configuration — the only cell you normally edit
Set your sport, point `VIDEO_PATH` at your match file, and (optionally) your ad PNG.
Leave `VIDEO_URL`/`VIDEO_PATH` as-is to run on an auto-generated synthetic clip.

In [ ]:
from pathlib import Path

# ---- INPUTS -----------------------------------------------------------
SPORT       = "cricket_test"          # cricket_test | cricket_t20 | soccer
VIDEO_PATH  = "./data/match.mp4"      # your match file goes here
VIDEO_URL   = ""                      # optional direct URL; if set, downloaded to VIDEO_PATH
AD_IMAGE_PATH = "./assets/ad_banner.png"   # auto-generated if missing
AD_STRATEGY = "auto"                  # auto | sightscreen | hoarding | ground
AD_BOARD_ASPECT = 8.0                 # width:height of the ad as placed on the boundary board
HOARDING_MODEL  = "hoarding_best.pt"  # trained detector; falls back to heuristic if absent

# ---- CAPTION MODEL (open, non-gated) ----------------------------------
CAPTION_MODEL    = "Qwen/Qwen2.5-3B-Instruct"
CAPTION_FALLBACK = "Qwen/Qwen2.5-1.5B-Instruct"

# ---- WHISPER ----------------------------------------------------------
WHISPER_MODEL   = "openai/whisper-large-v3"
WHISPER_FALLBACK = "openai/whisper-small"   # used if large-v3 OOMs / is too slow

# ---- OPTIONAL FEATURES ------------------------------------------------
USE_OCR        = False   # cricket scoreboard OCR (needs tesseract binary)
USE_OCCLUSION  = True    # keep players/ball IN FRONT of the ad (virtual replacement)

# ---- HIGHLIGHT SENSITIVITY (per sport) --------------------------------
HIGHLIGHT_SETTINGS = {
    "cricket_test": dict(min_height=0.55, min_distance=60, window_before=12, window_after=18),
    "cricket_t20":  dict(min_height=0.50, min_distance=20, window_before=8,  window_after=12),
    "soccer":       dict(min_height=0.50, min_distance=30, window_before=10, window_after=15),
}
H = HIGHLIGHT_SETTINGS[SPORT]

# ---- FUSION WEIGHTS ---------------------------------------------------
W_CROWD, W_SPEECH, W_VISION = 0.40, 0.30, 0.30

RESULTS = Path("./results"); RESULTS.mkdir(exist_ok=True)
Path("./data").mkdir(exist_ok=True); Path("./assets").mkdir(exist_ok=True)
print("Config loaded. Sport =", SPORT, "| settings =", H)


## 4 · Get a video — your file, a URL, or a synthetic fallback
This cell guarantees a valid MP4 exists at `VIDEO_PATH`. If your file isn't there and
no URL is given, it **synthesises a short clip** with two planted loud "roar" moments
so the whole pipeline is demonstrable offline.

In [ ]:
import os, numpy as np

def probe(path):
    r = sh(f'ffprobe -v error -show_entries format=duration,size '
           f'-of default=noprint_wrappers=1 "{path}"')
    return r.stdout.strip()

def make_synthetic_clip(path, seconds=90, fps=25, w=854, h=480):
    """Broadcast-style clip: crowd band on top, a boundary AD-BOARD strip,
    then the green field below. The board strip sits right above the field's
    top edge -- exactly where the real boundary hoardings are -- so the ad
    detector can be validated. Loud 'roar' moments planted at ~25s and ~65s."""
    import cv2
    tmp_v = "./data/_synth_video.mp4"
    vw = cv2.VideoWriter(tmp_v, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    total = seconds * fps
    crowd_h  = int(0.17 * h)              # crowd stand
    board_h  = int(0.055 * h)             # boundary ad board strip
    field_top = crowd_h + board_h         # where the green field starts
    roar = set(range(24*fps, 28*fps)) | set(range(64*fps, 68*fps))
    rng = np.random.default_rng(0)
    base = rng.integers(-25, 25, (crowd_h, w, 3))
    crowd_tex = np.clip(np.array([95, 90, 120]) + base, 0, 255).astype(np.uint8)  # brown/tan crowd (BGR)
    for i in range(total):
        f = np.zeros((h, w, 3), np.uint8)
        f[:crowd_h] = crowd_tex                                   # crowd
        f[crowd_h:field_top] = (45, 45, 55)                       # dark board base
        warm = [(40,40,210),(50,200,230),(210,210,210),(60,60,200)]  # reds/whites (BGR), not green
        for j, bx in enumerate(range(20, w-120, 150)):              # existing ads on board
            cv2.rectangle(f, (bx, crowd_h+4), (bx+110, field_top-4), warm[j % len(warm)], -1)
        f[field_top:] = (40, 120, 40)                             # field
        f[int(h*0.6):] = (35, 110, 35)
        cv2.rectangle(f, (w//2-40, field_top+30), (w//2+40, h-60), (90,150,90), -1)  # pitch
        bx = int(w*0.2 + (w*0.6) * (i % (fps*4)) / (fps*4))       # moving ball
        by = int(field_top + 60 + 40*np.sin(i/6.0))
        cv2.circle(f, (bx, by), 6, (255,255,255), -1)
        if i in roar:
            cv2.circle(f, (w//2, h//2), 55, (0,215,255), 6)
        vw.write(f)
    vw.release()
    sr = 16000
    aud = (np.random.randn(seconds*sr) * 0.03).astype(np.float32)
    for s in (25, 65):
        aud[s*sr:(s+3)*sr] += (np.random.randn(3*sr) * 0.5).astype(np.float32)
    aud = np.clip(aud, -1, 1)
    import soundfile as sf
    sf.write("./data/_synth_audio.wav", aud, sr)
    sh(f'ffmpeg -y -i {tmp_v} -i ./data/_synth_audio.wav -c:v libx264 '
       f'-pix_fmt yuv420p -c:a aac -shortest "{path}"')
    for t in (tmp_v, "./data/_synth_audio.wav"):
        try: os.remove(t)
        except OSError: pass

if VIDEO_URL:
    print("Downloading", VIDEO_URL)
    sh(f'wget -q -O "{VIDEO_PATH}" "{VIDEO_URL}"')

if not os.path.exists(VIDEO_PATH) or os.path.getsize(VIDEO_PATH) < 10_000:
    print("No usable video at", VIDEO_PATH, "-> generating a synthetic test clip.")
    make_synthetic_clip(VIDEO_PATH)

print("VIDEO READY:", VIDEO_PATH)
print(probe(VIDEO_PATH))


## 5 · A default ad image (auto-generated if you didn't supply one)
A transparent-background PNG composites most cleanly onto the surface.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

if not os.path.exists(AD_IMAGE_PATH):
    W, Hh = 800, 100                      # wide + thin, like a real boundary board ad
    img = Image.new("RGBA", (W, Hh), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    d.rounded_rectangle([0, 0, W-1, Hh-1], radius=10, fill=(15, 32, 90, 245))
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 56)
    except Exception:
        font = ImageFont.load_default()
    text = "YOUR BRAND"
    tw = d.textlength(text, font=font)
    d.text(((W-tw)/2, (Hh-56)/2 - 4), text, font=font, fill=(255, 215, 0, 255))
    img.save(AD_IMAGE_PATH)
    print("Generated wide boundary-style ad ->", AD_IMAGE_PATH)
else:
    print("Using your ad ->", AD_IMAGE_PATH)


## 6 · Imports

In [ ]:
import cv2, numpy as np, librosa, json, time, re, math
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
print("Imports OK")


## 7 · Stream A — crowd-noise energy
Extracts the audio, emphasises the 200–4000 Hz band (voices/roar), computes per-second
RMS energy, normalises to 0–1, and smooths so a single clap can't spike a highlight.

In [ ]:
def extract_crowd_energy(video_path, sr=16000, smooth_sec=5):
    wav = "./results/_audio.wav"
    sh(f'ffmpeg -y -i "{video_path}" -ac 1 -ar {sr} -vn "{wav}"')
    try:
        y, _ = librosa.load(wav, sr=sr)
    except Exception as e:
        print("No audio track / load failed:", e, "-> crowd stream = zeros")
        dur = float(sh(f'ffprobe -v error -show_entries format=duration '
                       f'-of csv=p=0 "{video_path}"').stdout.strip() or 1)
        return np.zeros(int(dur))
    # pre-emphasis (high-pass-ish) to favour voices/roar over PA bass rumble
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])
    hop = sr  # one value per second
    rms = librosa.feature.rms(y=y, frame_length=sr, hop_length=hop)[0]
    if rms.max() > 0:
        rms = rms / rms.max()
    k = max(1, smooth_sec)
    rms = np.convolve(rms, np.ones(k)/k, mode="same")
    return rms.astype(np.float32)

print("extract_crowd_energy() defined")


## 8 · Load Whisper (open model, lazy, with fallback)
`whisper-large-v3` is open (no gating). If it OOMs or is unavailable we fall back to
`whisper-small`. Loaded in-process via `transformers` — no server.

In [ ]:
import torch
from transformers import pipeline as hf_pipeline

whisper_pipe = None
def load_whisper():
    global whisper_pipe
    if whisper_pipe is not None:
        return whisper_pipe
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    for model_id in (WHISPER_MODEL, WHISPER_FALLBACK):
        try:
            print("Loading Whisper:", model_id)
            whisper_pipe = hf_pipeline(
                "automatic-speech-recognition", model=model_id,
                torch_dtype=dtype, device=0 if DEVICE == "cuda" else -1,
                chunk_length_s=30, return_timestamps=True,
            )
            print("Whisper ready:", model_id)
            return whisper_pipe
        except Exception as e:
            print("Failed", model_id, "->", str(e)[:160])
    print("Whisper unavailable -> commentary stream will be empty.")
    return None

load_whisper()


## 9 · Stream B — commentary excitement
Transcribes with timestamps, then scores each segment against a sport-specific keyword
dictionary (highest-weight keyword wins).

In [ ]:
KEYWORDS = {
 "cricket": {
    1.0: ["six","sixer","out","wicket","caught","bowled","lbw","stumped","run out","clean bowled"],
    0.9: ["incredible","unbelievable","century","hundred","hat trick","magnificent"],
    0.8: ["four","boundary","fifty","gone","huge"],
    0.6: ["drs","review","no ball","free hit","appeal"],
    0.3: ["wide","dot ball","single","defended"],
 },
 "soccer": {
    1.0: ["goal","scores","golazo","what a strike","top corner"],
    0.9: ["incredible","unbelievable","brilliant","stunning"],
    0.8: ["penalty","free kick","header","chance","saves"],
    0.6: ["corner","offside","var","yellow card","red card"],
    0.3: ["throw in","pass","midfield"],
 },
}
def _kw_set(sport): return KEYWORDS["soccer" if sport=="soccer" else "cricket"]

def score_text(text, sport):
    t = " " + text.lower() + " "
    best = 0.0
    for w, words in _kw_set(sport).items():
        if any(k in t for k in words):
            best = max(best, w)
    return best

def transcribe_and_score(video_path, sport):
    if whisper_pipe is None:
        return []
    wav = "./results/_audio.wav"
    if not os.path.exists(wav):
        sh(f'ffmpeg -y -i "{video_path}" -ac 1 -ar 16000 -vn "{wav}"')
    out = whisper_pipe(wav)
    segs = []
    for ch in out.get("chunks", []):
        ts = ch.get("timestamp") or (None, None)
        start, end = ts[0], ts[1]
        if start is None:
            continue
        txt = ch["text"].strip()
        segs.append(dict(start=float(start), end=float(end or start+1),
                         text=txt, score=score_text(txt, sport)))
    print(f"{len(segs)} commentary segments scored")
    return segs

print("transcribe_and_score() defined")


## 10 · Stream C — visual excitement (motion + scene cuts)
Real visual signal with **no fake label mapping**: per-second motion energy from frame
differencing, boosted at hard scene cuts (broadcasts cut to replays/celebrations right
after big moments).

In [ ]:
def score_frames(video_path, sample_fps=4):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    dur = int(total / fps) if total else 1
    step = max(1, int(round(fps / sample_fps)))
    per_sec = {}
    prev_gray = prev_hist = None
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            sec = int(idx / fps)
            g = cv2.cvtColor(cv2.resize(frame, (160, 90)), cv2.COLOR_BGR2GRAY)
            motion = 0.0; cut = 0.0
            if prev_gray is not None:
                motion = float(np.mean(cv2.absdiff(g, prev_gray))) / 255.0
                hist = cv2.calcHist([g], [0], None, [32], [0, 256]).flatten()
                hist /= (hist.sum() + 1e-6)
                if prev_hist is not None:
                    d = cv2.compareHist(prev_hist.astype("float32"),
                                        hist.astype("float32"), cv2.HISTCMP_BHATTACHARYYA)
                    cut = float(d)
                prev_hist = hist
            prev_gray = g
            per_sec.setdefault(sec, []).append(0.6*motion + 0.4*cut)
        idx += 1
    cap.release()
    scores = np.zeros(dur, np.float32)
    for s in range(dur):
        if s in per_sec: scores[s] = np.mean(per_sec[s])
    if scores.max() > 0: scores /= scores.max()
    return scores

print("score_frames() defined")


## 11 · Fusion + peak detection
Resample all three streams to one-value-per-second, weighted blend (0.4/0.3/0.3),
Gaussian smooth, then `find_peaks` with the sport's height/distance rules.

In [ ]:
def _resample(arr, n):
    arr = np.asarray(arr, np.float32)
    if len(arr) == 0: return np.zeros(n, np.float32)
    if len(arr) == n: return arr
    xp = np.linspace(0, 1, len(arr)); x = np.linspace(0, 1, n)
    return np.interp(x, xp, arr).astype(np.float32)

def fuse_modalities(crowd, commentary, vision, duration):
    n = max(duration, 1)
    crowd_r  = _resample(crowd,  n)
    vision_r = _resample(vision, n)
    speech = np.zeros(n, np.float32)
    for seg in commentary:
        a, b = int(seg["start"]), min(int(seg["end"]) + 1, n)
        speech[a:b] = np.maximum(speech[a:b], seg["score"])
    fused = W_CROWD*crowd_r + W_SPEECH*speech + W_VISION*vision_r
    if fused.max() > 0: fused /= fused.max()
    fused = gaussian_filter1d(fused, sigma=2)
    return fused, dict(crowd=crowd_r, speech=speech, vision=vision_r)

def detect_highlights(fused, commentary):
    peaks, props = find_peaks(fused, height=H["min_height"], distance=H["min_distance"])
    out = []
    for p in peaks:
        seg_txt = next((s["text"] for s in commentary
                        if s["start"] <= p <= s["end"]), "")
        out.append(dict(peak_sec=int(p),
                        clip_start=max(0, int(p)-H["window_before"]),
                        clip_end=int(p)+H["window_after"],
                        peak_score=round(float(fused[p]), 3),
                        commentary=seg_txt))
    out.sort(key=lambda d: d["peak_score"], reverse=True)
    print(f"Found {len(out)} highlights")
    for h in out:
        m, s = divmod(h["peak_sec"], 60)
        print(f"  {m:02d}:{s:02d}  score={h['peak_score']}  '{h['commentary'][:40]}'")
    return out

print("fusion + detection defined")


## 12 · ffmpeg clip helpers (extract / caption overlay / concatenate)

In [ ]:
def extract_clip(src, start, end, out_path):
    sh(f'ffmpeg -y -ss {start} -to {end} -i "{src}" -c:v libx264 '
       f'-pix_fmt yuv420p -c:a aac "{out_path}"')
    return out_path

def _esc(t):  # escape text for ffmpeg drawtext
    return t.replace("\\", "").replace(":", r"\:").replace("'", "")[:80]

def add_caption_overlay(src, out_path, title, score):
    title = _esc(title)
    badge = f"EXCITEMENT {score:.2f}"
    vf = (f"drawtext=text='{title}':fontcolor=white:fontsize=26:box=1:"
          f"boxcolor=black@0.5:boxborderw=12:x=(w-text_w)/2:y=h-60,"
          f"drawtext=text='{badge}':fontcolor=yellow:fontsize=20:box=1:"
          f"boxcolor=black@0.5:boxborderw=8:x=20:y=20")
    r = sh(f'ffmpeg -y -i "{src}" -vf "{vf}" -c:a copy "{out_path}"')
    if r.returncode != 0:      # font missing etc -> keep uncaptioned clip
        sh(f'cp "{src}" "{out_path}"')
    return out_path

def concatenate_clips(clip_paths, out_path):
    lst = "./results/_concat.txt"
    with open(lst, "w") as f:
        for c in clip_paths:
            f.write(f"file '{os.path.abspath(c)}'\n")
    sh(f'ffmpeg -y -f concat -safe 0 -i "{lst}" -c:v libx264 -pix_fmt yuv420p -c:a aac "{out_path}"')
    return out_path

print("ffmpeg helpers defined")


## 13 · Virtual ad insertion — surface detection
Finds candidate surfaces in a frame: **sight screen** (large white panel), **hoarding**
(wide rectangle), **ground** (large uniform green region). Returns best-first.

In [ ]:
def _field_mask(frame):
    """Largest green region in the lower frame."""
    h, w = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, (30, 35, 35), (95, 255, 255))
    green = cv2.morphologyEx(green, cv2.MORPH_OPEN,  np.ones((5, 5), np.uint8))
    green = cv2.morphologyEx(green, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    num, labels, stats, cent = cv2.connectedComponentsWithStats(green, 8)
    best, ba = None, 0
    for i in range(1, num):
        a = stats[i, cv2.CC_STAT_AREA]
        if a > ba and cent[i, 1] > 0.40 * h and a > 0.10 * w * h:
            best, ba = i, a
    if best is None:
        return None, h, w
    return (labels == best).astype(np.uint8), h, w


def _row_boardness(frame, x_lo, x_hi, y):
    """Score for 'does this row look like an ad board': colour spread (multiple
    coloured panels side by side) + edge density (panel borders / logos / text),
    both of which a plain crowd stand or grass do not have."""
    h = frame.shape[0]
    if y < 0 or y >= h or x_hi - x_lo < 8:
        return 0.0
    seg = frame[y, x_lo:x_hi].astype(np.float32)
    color_std = float(seg.std(axis=0).mean())
    gray = cv2.cvtColor(seg.reshape(1, -1, 3).astype(np.uint8), cv2.COLOR_BGR2GRAY).flatten().astype(np.float32)
    edges = np.abs(np.diff(gray))
    edge_density = float((edges > 22).mean())
    return color_std * 0.6 + edge_density * 140 * 0.4


def detect_boundary_board(frame, board_aspect=8.0, default_frac=0.05):
    """Heuristic boundary-hoarding detector (classical CV, no training needed).
    Finds the field's top edge, then MEASURES the actual board thickness right
    there (colour-spread + edge-density profile) instead of guessing a fixed
    fraction of frame height -- this is what makes the vertical placement land
    on the real board instead of drifting into the crowd or the grass."""
    field, h, w = _field_mask(frame)
    if field is None:
        return None
    tops = np.full(w, -1, np.int32)
    for x in range(0, w, 2):
        colpix = field[:, x] > 0
        if colpix.sum() < 0.10 * h:
            continue
        ys = np.where(colpix)[0]
        y = int(ys.max())
        while y - 1 >= 0 and colpix[y - 1]:
            y -= 1
        tops[x] = y
    valid = np.where(tops >= 0)[0]
    if len(valid) < w * 0.15:
        return None
    ys = tops[valid].astype(np.float32)
    med = float(np.median(ys))
    if med < 0.06 * h:                  # field touches frame top -> no boundary in view
        return None
    keep = valid[np.abs(ys - med) < 0.08 * h]
    if len(keep) < 10:
        keep = valid
    x_lo, x_hi = int(keep.min()), int(keep.max())
    field_w = max(1, x_hi - x_lo)

    # --- measure the real board band (colour-spread + edge profile upward from
    # the field edge), tolerant of a small leading gap (anti-aliasing / trim) ---
    max_search = max(10, int(0.15 * h))
    rope = int(med)
    scores = np.array([_row_boardness(frame, x_lo, x_hi, rope - d) for d in range(max_search)])
    measured = 0
    if scores.max() > 1e-6:
        norm = scores / scores.max()
        hits = np.where(norm > 0.22)[0]
        if len(hits) > 0:
            max_gap = 3
            clusters, cur = [], [hits[0]]
            for d in hits[1:]:
                if d - cur[-1] <= max_gap:
                    cur.append(d)
                else:
                    clusters.append(cur)
                    cur = [d]
            clusters.append(cur)
            measured = min(clusters, key=lambda c: c[0])[-1] + 1
    min_band, max_band = 0.018 * h, 0.14 * h
    band = measured if (min_band <= measured <= max_band) else max(14, int(default_frac * h))

    ad_w = int(min(field_w * 0.85, band * board_aspect))
    cx = (x_lo + x_hi) // 2
    ax_lo, ax_hi = cx - ad_w // 2, cx + ad_w // 2

    def top_at(x):
        x = int(np.clip(x, 0, w - 1))
        return int(tops[x]) if tops[x] >= 0 else int(med)
    yl, yr, gap = top_at(ax_lo), top_at(ax_hi), 2
    cl = lambda v: float(max(0, v))
    quad = np.float32([[ax_lo, cl(yl-band-gap)], [ax_hi, cl(yr-band-gap)],
                       [ax_hi, cl(yr-gap)],      [ax_lo, cl(yl-gap)]])
    return ("hoarding", 0.95, quad)


# ---- Trained hoarding detector (OPTIONAL -- only used if you later drop a
# hoarding_best.pt file next to this notebook; harmless no-op otherwise) ----
_HOARD_MODEL = "unloaded"
def _load_hoard():
    global _HOARD_MODEL
    if _HOARD_MODEL == "unloaded":
        if os.path.exists(HOARDING_MODEL):
            from ultralytics import YOLO
            _HOARD_MODEL = YOLO(HOARDING_MODEL)
            print("Loaded trained hoarding detector:", HOARDING_MODEL)
        else:
            _HOARD_MODEL = None
    return _HOARD_MODEL

def detect_hoarding_yolo(frame, conf=0.35):
    m = _load_hoard()
    if m is None:
        return None
    r = m(frame, verbose=False, conf=conf)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None
    b = r.boxes[int(r.boxes.conf.argmax())]
    x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].cpu().numpy()]
    quad = np.float32([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
    return ("hoarding", float(b.conf[0]), quad)


def detect_surfaces(frame):
    """HOARDING ONLY. Trained model first (if present), classical CV heuristic
    second. Neither found -> [] so the clip runs WITHOUT an ad."""
    try:
        y = detect_hoarding_yolo(frame)
        if y is not None:
            return [y]
    except Exception as e:
        print("yolo hoarding skipped:", str(e)[:80])
    h = detect_boundary_board(frame, AD_BOARD_ASPECT)
    return [h] if h is not None else []

print("detect_surfaces() = measured-thickness heuristic (+ optional trained model)")


## 14 · Virtual ad insertion — perspective warp + alpha blend
Homography-warps the ad onto the 4 surface corners, matches brightness, alpha-blends.

In [ ]:
def load_ad(path):
    ad = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if ad is None:
        raise FileNotFoundError(f"Cannot read ad image: {path}")
    if ad.shape[2] == 3:  # add opaque alpha
        ad = np.dstack([ad, np.full(ad.shape[:2], 255, np.uint8)])
    return ad

def warp_ad_to_surface(frame, ad, dst_pts):
    """Perspective-warp the ad onto the board quad and blend it so it looks printed
    on: soft per-channel lighting match, matched film grain, feathered edges."""
    h, w = frame.shape[:2]
    ah, aw = ad.shape[:2]
    src = np.float32([[0, 0], [aw, 0], [aw, ah], [0, ah]])
    Hm = cv2.getPerspectiveTransform(src, dst_pts.astype(np.float32))
    warped = cv2.warpPerspective(ad, Hm, (w, h), flags=cv2.INTER_LINEAR)
    bgr = warped[:, :, :3].astype(np.float32)
    alpha = (warped[:, :, 3].astype(np.float32) / 255.0)

    m = alpha > 0.1
    if m.sum() > 20:
        for c in range(3):                          # soft lighting/colour match
            fm = frame[:, :, c][m].mean()
            am = bgr[:, :, c][m].mean() + 1e-6
            bgr[:, :, c] *= np.clip((fm / am) ** 0.5, 0.6, 1.6)
        bgr = np.clip(bgr, 0, 255)
        amp = float(np.clip(frame[m].std() * 0.10, 2, 12))   # matched grain
        bgr += (np.random.randn(h, w, 1).astype(np.float32) * amp) * alpha[..., None]
        bgr = np.clip(bgr, 0, 255)

    alpha = cv2.GaussianBlur(alpha, (0, 0), 1.2)[:, :, None]  # feather edges
    out = frame.astype(np.float32) * (1 - alpha) + bgr * alpha
    return out.astype(np.uint8)

print("warp_ad_to_surface() with photometric blend defined")


## 15 · Virtual ad insertion — optical-flow tracker
Lucas-Kanade tracks the surface as the camera pans, with a forward-backward check so a
bad track is dropped instead of dragging the ad off-surface.

In [ ]:
class AdTracker:
    def __init__(self, gray, corners):
        self.corners = corners.astype(np.float32)
        self.prev = gray
        x0,y0 = corners.min(0); x1,y1 = corners.max(0)
        roi = gray[int(y0):int(y1), int(x0):int(x1)]
        pts = cv2.goodFeaturesToTrack(roi, 60, 0.01, 5) if roi.size else None
        self.pts = (pts.reshape(-1,2) + [x0,y0]).astype(np.float32) if pts is not None else None

    def update(self, gray):
        if self.pts is None or len(self.pts) < 6:
            self.prev = gray; return self.corners
        lk = dict(winSize=(21,21), maxLevel=3,
                  criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 30, 0.01))
        nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, self.pts, None, **lk)
        back, st2, _ = cv2.calcOpticalFlowPyrLK(gray, self.prev, nxt, None, **lk)
        good = (np.abs(self.pts - back).max(1) < 1.5) & st.flatten().astype(bool)
        if good.sum() >= 6:
            old = self.pts[good]; new = nxt[good]
            M, _ = cv2.estimateAffinePartial2D(old, new)
            if M is not None:
                self.corners = cv2.transform(self.corners[None], M)[0]
                self.pts = new
        self.prev = gray
        return self.corners

print("AdTracker defined")


## 16 · Virtual ad insertion — per-clip pipeline
Picks the best surface (respecting `AD_STRATEGY`), tracks it across frames, warps+blends
the ad each frame, optionally restores player pixels (occlusion), writes the output clip.

In [ ]:
_seg = None
def _get_seg():
    """Segmentation model for occlusion masks (persons + sports ball)."""
    global _seg
    if _seg is None:
        from ultralytics import YOLO
        _seg = YOLO("yolov8n-seg.pt")
    return _seg

def _occlusion_mask(seg, frame):
    h, w = frame.shape[:2]
    res = seg(frame, verbose=False, classes=[0, 32])[0]   # 0=person, 32=sports ball
    person = np.zeros((h, w), np.uint8)
    if res.masks is not None:
        for poly in res.masks.xy:
            if len(poly) >= 3:
                cv2.fillPoly(person, [poly.astype(np.int32)], 1)
    return person

def _consensus_surface(cap, n, samples=5):
    """Detect the board on several frames spread across the clip and return the
    MEDIAN quad among frames that agree -- one occluded/blurry frame (a player
    crossing the boundary, a fast pan) can no longer drag the whole clip's
    placement off. Returns (name, quad) or (None, None) if no majority found."""
    if n <= 0:
        return None, None
    fracs = [0.05, 0.25, 0.45, 0.65, 0.85][:samples]
    idxs = sorted(set(int(f * max(n - 1, 0)) for f in fracs))
    found = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok:
            continue
        cands = detect_surfaces(fr)
        if cands:
            found.append(cands[0])
    print(f"  hoarding consensus: {len(found)}/{len(idxs)} sampled frames found a board")
    if len(found) < max(1, len(idxs) // 2 + len(idxs) % 2):   # need a majority
        return None, None
    quads = np.stack([f[2] for f in found])
    med_quad = np.median(quads, axis=0).astype(np.float32)
    return found[0][0], med_quad


def insert_ad_in_clip(clip_path, ad_path, out_path, strategy="auto"):
    ad = load_ad(ad_path)
    cap = cv2.VideoCapture(clip_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    surf_name, corners = _consensus_surface(cap, n)

    # HOARDING-ONLY: no majority board detection -> keep clip as-is, no ad
    if corners is None:
        cap.release(); sh(f'cp "{clip_path}" "{out_path}"')
        print("  no hoarding consensus -> clip kept without ad")
        return {"surface": None}

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ok, first = cap.read()
    tracker = AdTracker(cv2.cvtColor(first, cv2.COLOR_BGR2GRAY), corners)
    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    seg = _get_seg() if USE_OCCLUSION else None

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    while True:
        ok, frame = cap.read()
        if not ok: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        cur = tracker.update(gray)
        comp = warp_ad_to_surface(frame, ad, cur)
        if seg is not None:                       # keep players/ball in front of the ad
            mask = _occlusion_mask(seg, frame)
            comp[mask > 0] = frame[mask > 0]
        vw.write(comp)
    cap.release(); vw.release()

    tmp = out_path + ".tmp.mp4"
    sh(f'ffmpeg -y -i "{out_path}" -i "{clip_path}" -map 0:v:0 -map 1:a:0? '
       f'-c:v libx264 -pix_fmt yuv420p -c:a aac -shortest "{tmp}" '
       f'&& mv "{tmp}" "{out_path}"')
    return {"surface": surf_name}

print("insert_ad_in_clip() = multi-frame consensus + virtual replacement + occlusion")


## 17 · Caption model — open LLM, in-process (no vLLM server)
Loads `Qwen2.5-3B-Instruct` via `transformers`. Falls back to 1.5B, then to a template
if no model can load. This replaces the entire fragile vLLM-server section.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

caption_model = caption_tok = None
def load_caption_model():
    global caption_model, caption_tok
    if caption_model is not None or caption_model is False:
        return
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    for mid in (CAPTION_MODEL, CAPTION_FALLBACK):
        try:
            print("Loading caption LLM:", mid)
            caption_tok = AutoTokenizer.from_pretrained(mid)
            caption_model = AutoModelForCausalLM.from_pretrained(
                mid, torch_dtype=dtype,
                device_map="auto" if DEVICE == "cuda" else None)
            print("Caption LLM ready:", mid)
            return
        except Exception as e:
            print("Failed", mid, "->", str(e)[:160])
    caption_model = False   # signal: use template fallback
    print("No caption LLM -> template captions will be used.")

load_caption_model()


## 18 · Caption generation (strict JSON, safe fallback)

In [ ]:
def _template_caption(sport, sec, score, commentary):
    m, s = divmod(int(sec), 60)
    return {"title": f"Big Moment at {m:02d}:{s:02d}",
            "caption": f"What a passage of play! {commentary[:60]} "
                       f"#Highlights #{sport.replace('_','').title()}"}

def generate_caption(sport, sec, score, commentary):
    if not caption_model:
        return _template_caption(sport, sec, score, commentary)
    prompt = (f"You write punchy social-media highlight captions for {sport} clips.\n"
              f"Moment at {int(sec)}s, excitement {score:.2f}/1.0. "
              f"Commentary heard: \"{commentary[:200]}\".\n"
              f"Reply with ONLY JSON: {{\"title\": \"...\", \"caption\": \"... with #hashtags\"}}")
    try:
        msgs = [{"role": "user", "content": prompt}]
        text = caption_tok.apply_chat_template(msgs, tokenize=False,
                                               add_generation_prompt=True)
        inp = caption_tok(text, return_tensors="pt").to(caption_model.device)
        out = caption_model.generate(**inp, max_new_tokens=120, do_sample=True,
                                     temperature=0.7, top_p=0.9)
        resp = caption_tok.decode(out[0][inp.input_ids.shape[1]:],
                                  skip_special_tokens=True)
        m = re.search(r"\{.*\}", resp, re.S)
        data = json.loads(m.group(0))
        return {"title": str(data.get("title", "Highlight"))[:80],
                "caption": str(data.get("caption", ""))[:160]}
    except Exception as e:
        print("caption fallback:", str(e)[:80])
        return _template_caption(sport, sec, score, commentary)

print("generate_caption() defined")


## 19 · (Optional) cricket scoreboard OCR boost
Near-perfect signal for cricket: a score change of +4/+6 or a wicket is a guaranteed
event. Boosts the fused timeline at those exact seconds. Skipped unless `USE_OCR=True`
and tesseract is installed.

In [ ]:
def ocr_event_boost(video_path, fused, sport):
    if not (USE_OCR and sport.startswith("cricket")):
        return fused
    try:
        import pytesseract
    except Exception:
        print("pytesseract not available -> skipping OCR boost"); return fused
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    prev = None; idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % int(fps*2) == 0:
            h, w = frame.shape[:2]
            roi = frame[int(h*0.85):, :int(w*0.35)]   # bottom-left scorebar
            g = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
            g = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
            txt = pytesseract.image_to_string(g, config="--psm 7")
            m = re.search(r"(\d{1,3})\s*/\s*(\d)", txt)
            if m:
                runs, wkts = int(m.group(1)), int(m.group(2))
                sec = int(idx/fps)
                if prev:
                    if wkts > prev[1] or (runs-prev[0]) in (4, 6):
                        lo, hi = max(0, sec-3), min(len(fused), sec+3)
                        fused[lo:hi] = np.minimum(1.0, fused[lo:hi] + 0.40)
                prev = (runs, wkts)
        idx += 1
    cap.release()
    if fused.max() > 0: fused /= fused.max()
    return fused

print("ocr_event_boost() defined")


## 20 · Master pipeline

In [ ]:
def generate_highlight_reel():
    t0 = time.time()
    dur = int(float(sh(f'ffprobe -v error -show_entries format=duration '
                       f'-of csv=p=0 "{VIDEO_PATH}"').stdout.strip() or 1))
    print(f"Video duration ~{dur}s\n--- Stream A: crowd ---")
    crowd = extract_crowd_energy(VIDEO_PATH)
    print("--- Stream B: commentary ---")
    commentary = transcribe_and_score(VIDEO_PATH, SPORT)
    print("--- Stream C: vision ---")
    vision = score_frames(VIDEO_PATH)
    print("--- Fusion ---")
    fused, parts = fuse_modalities(crowd, commentary, vision, dur)
    fused = ocr_event_boost(VIDEO_PATH, fused, SPORT)
    highlights = detect_highlights(fused, commentary)

    final_clips, meta = [], []
    for i, hl in enumerate(highlights):
        print(f"\n[Highlight {i}] {hl['peak_sec']}s score={hl['peak_score']}")
        raw = extract_clip(VIDEO_PATH, hl["clip_start"], hl["clip_end"],
                           f"./results/clip_{i:02d}_raw.mp4")
        ad_clip = f"./results/clip_{i:02d}_ad.mp4"
        info = insert_ad_in_clip(raw, AD_IMAGE_PATH, ad_clip, AD_STRATEGY)
        cap = generate_caption(SPORT, hl["peak_sec"], hl["peak_score"], hl["commentary"])
        final = add_caption_overlay(ad_clip, f"./results/clip_{i:02d}_final.mp4",
                                    cap["title"], hl["peak_score"])
        final_clips.append(final)
        meta.append(dict(index=i, **hl, surface=info.get("surface"), **cap))

    reel = None
    if final_clips:
        reel = concatenate_clips(final_clips, "./results/highlight_reel.mp4")
    json.dump({"sport": SPORT, "duration_sec": dur,
               "signals": {k: v.tolist() for k, v in parts.items()},
               "fused": fused.tolist(), "highlights": meta},
              open("./results/metadata.json", "w"), indent=2)
    print(f"\nDONE in {time.time()-t0:.0f}s -> {reel}")
    return reel, meta, fused, parts

print("generate_highlight_reel() defined")


## 21 · Run the pipeline

In [ ]:
reel, meta, fused, parts = generate_highlight_reel()
print("\nFinal reel:", reel)
print("Highlights:", len(meta))


## 22 · Excitement timeline chart (Slide 3 of your deck)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
t = np.arange(len(fused))
ax.plot(t, parts["crowd"],  label="Crowd (A)", alpha=.5)
ax.plot(t, parts["speech"], label="Commentary (B)", alpha=.5)
ax.plot(t, parts["vision"], label="Vision (C)", alpha=.5)
ax.plot(t, fused, label="Fused", color="black", lw=2.2)
for h in meta:
    ax.axvline(h["peak_sec"], color="red", ls="--", alpha=.6)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Excitement (0-1)")
ax.set_title(f"Multimodal Excitement Timeline — {SPORT}")
ax.legend(loc="upper right")
plt.tight_layout(); plt.savefig("./results/timeline.png", dpi=130)
plt.show()
print("Saved ./results/timeline.png")


## 23 · GPU / latency metrics (Slide 4 of your deck)

In [ ]:
print("=== AMD GPU metrics ===")
r = sh("rocm-smi --showmeminfo vram --showuse --showpower")
print(r.stdout if r.returncode == 0 else "rocm-smi not available in this runtime")
try:
    import torch
    if torch.cuda.is_available():
        print(f"torch peak VRAM allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
        print("Device:", torch.cuda.get_device_name(0))
except Exception as e:
    print(e)

print("\nModels used:")
print("  Whisper          :", WHISPER_MODEL, "(fallback", WHISPER_FALLBACK + ")")
print("  Caption LLM      :", CAPTION_MODEL, "(fallback", CAPTION_FALLBACK + ")")
print("  Visual signal    : OpenCV motion + scene-cut (no GPU model)")
print("  Ad insertion     : OpenCV homography + Lucas-Kanade optical flow")
print("  Player occlusion :", "YOLOv8n" if USE_OCCLUSION else "off")
